In [1]:
import pandas as pd
import numpy as np

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Load reorder recommendations output from Phase 12
rec_path = '../data/processed/reorder_recommendations.csv'
df = pd.read_csv(rec_path)

print(f"Loaded {len(df):,} SKU decision records for risk classification.")

Loaded 1,676 SKU decision records for risk classification.


In [2]:
# Rule-Based Risk Engine
def classify_inventory_risk(row):
    inv_pos = row['Inventory_Position']
    rop = row['Reorder_Point']
    ss = row['Safety_Stock']
    
    if inv_pos <= rop:
        return 'HIGH STOCKOUT RISK', 'REORDER NOW', 1
    elif inv_pos <= (rop + ss):
        return 'MODERATE RISK', 'MONITOR', 2
    elif inv_pos <= (rop * 2.5):
        return 'SUFFICIENT STOCK', 'SUFFICIENT STOCK', 4
    else:
        return 'POTENTIAL OVERSTOCK', 'POTENTIAL OVERSTOCK', 3

# Apply classification rules
risk_results = df.apply(classify_inventory_risk, axis=1)

df['Risk_Category'] = [r[0] for r in risk_results]
df['Risk_Action'] = [r[1] for r in risk_results]
df['Priority_Rank'] = [r[2] for r in risk_results]

# Sort by operational urgency (Priority Rank 1 first, then by highest demand volume)
prioritized_df = df.sort_values(
    by=['Priority_Rank', 'Forecast_Demand'], 
    ascending=[True, False]
).reset_index(drop=True)

print("=== RISK CLASSIFICATION SUMMARY ===")
print(prioritized_df['Risk_Category'].value_counts())

=== RISK CLASSIFICATION SUMMARY ===
Risk_Category
HIGH STOCKOUT RISK     697
MODERATE RISK          623
SUFFICIENT STOCK       332
POTENTIAL OVERSTOCK     24
Name: count, dtype: int64


In [3]:
# Format visual output table for supply chain decision-makers
output_cols = [
    'StockCode', 'Forecast_Demand', 'Safety_Stock', 
    'Reorder_Point', 'Inventory_Position', 'Reorder_Quantity', 
    'Risk_Category', 'Risk_Action'
]

prioritized_table = prioritized_df[output_cols]

print("\n=== TOP 10 URGENT ACTION ITEMS (REORDER NOW) ===")
display(prioritized_table[prioritized_table['Risk_Action'] == 'REORDER NOW'].head(10))

print("\n=== SAMPLE POTENTIAL OVERSTOCK ITEMS ===")
display(prioritized_table[prioritized_table['Risk_Action'] == 'POTENTIAL OVERSTOCK'].head(5))


=== TOP 10 URGENT ACTION ITEMS (REORDER NOW) ===


,StockCode,Forecast_Demand,Safety_Stock,Reorder_Point,Inventory_Position,Reorder_Quantity,Risk_Category,Risk_Action
0,22197,2955.75,2186.00,8097.50,3544.00,8279.00,HIGH STOCKOUT RISK,REORDER NOW
1,22086,1496.00,1427.00,4419.00,2906.00,3078.00,HIGH STOCKOUT RISK,REORDER NOW
2,20668,1015.00,820.00,2850.00,2358.00,1702.00,HIGH STOCKOUT RISK,REORDER NOW
3,22578,1009.75,908.00,2927.50,2393.00,1646.00,HIGH STOCKOUT RISK,REORDER NOW
4,84077,876.75,2349.00,4102.50,2463.00,1044.00,HIGH STOCKOUT RISK,REORDER NOW
5,22577,870.25,813.00,2553.50,1765.00,1716.00,HIGH STOCKOUT RISK,REORDER NOW
6,22910,757.50,846.00,2361.00,1155.00,1875.00,HIGH STOCKOUT RISK,REORDER NOW
7,22952,711.50,1660.00,3083.00,2671.00,175.00,HIGH STOCKOUT RISK,REORDER NOW
8,22355,630.50,389.00,1650.00,1087.00,1435.00,HIGH STOCKOUT RISK,REORDER NOW
9,21137,628.25,570.00,1826.50,943.00,1570.00,HIGH STOCKOUT RISK,REORDER NOW



=== SAMPLE POTENTIAL OVERSTOCK ITEMS ===


,StockCode,Forecast_Demand,Safety_Stock,Reorder_Point,Inventory_Position,Reorder_Quantity,Risk_Category,Risk_Action
1320,21668,35.50,184.00,255.00,666.00,0.00,POTENTIAL OVERSTOCK,POTENTIAL OVERSTOCK
1321,21672,33.00,192.00,258.00,715.00,0.00,POTENTIAL OVERSTOCK,POTENTIAL OVERSTOCK
1322,22052,12.50,98.00,123.00,358.00,0.00,POTENTIAL OVERSTOCK,POTENTIAL OVERSTOCK
1323,21720,8.75,57.00,74.50,191.00,0.00,POTENTIAL OVERSTOCK,POTENTIAL OVERSTOCK
1324,22855,8.25,189.00,205.50,537.00,0.00,POTENTIAL OVERSTOCK,POTENTIAL OVERSTOCK


In [ ]:
output_csv = '../data/processed/inventory_risk_prioritization.csv'
prioritized_table.to_csv(output_csv, index=False)
print(f"Risk prioritization table saved successfully to: {output_csv}")